# 03 — Revisão de leakage e criação de features

Esta etapa começa no arquivo limpo produzido por 02_Analise_Dados.ipynb. Ela não reexecuta a limpeza, não altera dataset_limpo.csv e não remove linhas. A matriz de modelagem sai separada em dataset_features.csv.

## Momento da previsão

Assumimos que a previsão é feita no grid, depois dos treinos e da classificação, imediatamente antes da largada. Por isso, grid, classificação e posições de treino podem ser features. Se o produto precisar prever antes do fim de semana, essas colunas devem sair.

O alvo principal é a ordem oficial completa do resultado (rd_position_display_order). rd_position_number fica ausente para DNF/DNS/DSQ e não cobre todos os participantes; a ordem de exibição preserva essas posições finais. Também mantemos a posição classificada e a vitória como alvos alternativos.

A divisão é temporal: treino até 2024, validação em 2025 e teste em 2026. As features históricas de 2025/2026 podem usar resultados de GPs anteriores da mesma temporada, pois representam previsões renovadas corrida a corrida. Não é uma previsão da temporada inteira feita antes da primeira etapa.

In [ ]:
from pathlib import Path

import pandas as pd

from feature_engineering import (
    FEATURE_COLUMNS,
    INPUT_PATH,
    OUTPUT_PATH,
    TARGET_COLUMNS,
    audit_source_columns,
    build_feature_frame,
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"{INPUT_PATH.name} não existe. Rode 02_Analise_Dados.ipynb primeiro."
    )

source = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Fonte limpa: {INPUT_PATH.name}")
print(f"Dimensões: {source.shape[0]:,} linhas × {source.shape[1]:,} colunas")
print(f"Linhas sem posição classificada: {source['rd_position_number'].isna().sum():,}")
print("Nenhuma linha é removida nesta etapa.")

## Auditoria da fonte

As colunas de standings dstand_* e cstand_* são atualizadas após a corrida. Os campos rd_* contêm resultados da etapa, então só os campos pré-largada escolhidos explicitamente passam para a matriz. Totais atuais de carreira e melhores marcas podem carregar resultados futuros em linhas antigas. Agregações de motor/chassi sem data confiável e campos do calendário com cobertura parcial também ficam fora.

In [ ]:
audit = audit_source_columns(source.columns.tolist())
print(f"Colunas sinalizadas para não usar diretamente: {len(audit)}")
audit_summary = (
    audit.groupby("decisão")
    .agg(
        quantidade=("coluna", "size"),
        exemplos=("coluna", lambda values: ", ".join(values.head(6))),
    )
    .sort_values("quantidade", ascending=False)
)
print(audit_summary.to_string())


## Features pré-largada

O código reutilizável está em feature_engineering.py. Ele cria históricos do piloto, do piloto naquele circuito e do construtor com shift(1), para usar apenas etapas anteriores. O construtor é agregado por corrida antes do cálculo do histórico, evitando que o resultado atual de um companheiro apareça na feature do outro.

In [ ]:
features = build_feature_frame(source)
features.to_csv(OUTPUT_PATH, index=False)

print(f"Features: {len(FEATURE_COLUMNS)}")
print(f"Alvos: {TARGET_COLUMNS}")
print(f"Linhas preservadas: {len(features):,}")
print(f"Arquivo gerado: {OUTPUT_PATH.name}")

split_summary = (
    features.groupby("dataset_split")
    .agg(
        linhas=("rd_race_id", "size"),
        corridas=("rd_race_id", "nunique"),
        primeira_corrida=("race_date", "min"),
        ultima_corrida=("race_date", "max"),
        pilotos=("rd_driver_id", "nunique"),
    )
)
print(split_summary.to_string())

coverage_columns = [
    "grid_position", "qualifying_position", "fp1_position", "fp2_position",
    "fp3_position", "driver_prior_avg_finish_last5",
    "driver_circuit_prior_win_rate", "constructor_prior_win_rate",
]
coverage = (
    features.groupby("dataset_split")[coverage_columns]
    .apply(lambda frame: frame.notna().mean().mul(100).round(1))
    .T.rename_axis("feature")
)
print(coverage.to_string())


## Uso e limites

Use FEATURE_COLUMNS como entradas e os alvos separadamente. IDs de piloto, construtor e circuito são categorias, não quantidades. A matriz conserva os nulos das sessões; qualquer imputação ou codificação deve ser ajustada só no treino.

O arquivo anexado tem oito corridas de 2026, então o teste é parcial. As métricas correspondem a uma previsão renovada antes de cada GP, usando resultados conhecidos das etapas anteriores.

Os históricos começam em 2003, início do recorte. Portanto, as primeiras temporadas têm histórico truncado em relação à carreira real dos pilotos e construtores. Para a matriz, nulos de sessão continuam nulos; somente pontos ausentes são interpretados como zero ao construir o histórico de pontos, sem alterar dataset_limpo.csv.